<a href="https://colab.research.google.com/github/lucaslimb/agentic-ai-nano/blob/main/agentic_ai_nano.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LangGraph

In [ ]:
# Installando dependencias
!pip install langchain -q
!pip install langchain-core -q
!pip install langchain-community -q
!pip install langchain-ollama -q
!pip install langgraph -q

!pip install transformers

!pip install zstd

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conflicts
    package_set, _ = create_package_set_from_installed()
              

KeyboardInterrupt: 

In [ ]:
# Importando
import langgraph
import numpy as np
import pandas as pd

In [4]:
# Install zstd dependency
!sudo apt-get update && sudo apt-get install -y zstd

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [90.8 kB]
Get:10 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,995 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 

In [6]:
# Install e execute o modelo no background
!curl -fsSL https://ollama.com/install.sh | sh
!nohup ollama serve > /dev/null 2>&1 &
!ollama pull lama3:8b

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.

Error: pull model manifest: file does not exist


In [ ]:
from langchain_community.llms import Ollama

def processamento_resposta(pergunta):
  # Initialize the Ollama LLM
  llm = Ollama(model="gpt-oss:20b",num_predict=1024)

  # Send a prompt
  resposta = llm.invoke(pergunta)
  return resposta

processamento_resposta("Diga: Hello World!")

'Hello World!'

In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_community.llms import Ollama
from typing_extensions import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage


# Inicializar o modelo Ollama
llm = Ollama(model="gpt-oss:20b", num_predict=1024)


# Definir o esquema de estado
class State(TypedDict):
     messages: Annotated[list[str], add_messages]

# Definir a função do nó de entrada
def entrada_usuario(state: State) -> State:
    pergunta = input("Pergunta: ")
    return {"messages": [pergunta]}

# Definir a função do nó de processamento
def processamento_resposta(state: State) -> State:
    pergunta = state["messages"][-1]
    resposta = llm.invoke(pergunta)
    return {"messages": [f"A resposta para a sua pergunta '{pergunta}' é: {resposta}"]}

# Definir a função do nó de saída
def saida_resposta(state: State) -> State:
    print(state["messages"][-1])
    return state



def processamento_resposta(state: State) -> State:
    pergunta = state["messages"][-1]
    if isinstance(pergunta, str) and pergunta.strip():
        resposta = llm.invoke(pergunta)
        return {"messages": [f"A resposta para a sua pergunta '{pergunta}' é: {resposta}"]}
    else:
        raise ValueError("A pergunta deve ser uma string não vazia.")


# Criar o grafo de estado
grafo = StateGraph(State)

# Adicionar nós ao grafo
grafo.add_node("entrada", entrada_usuario)
grafo.add_node("processamento", processamento_resposta)
grafo.add_node("saida", saida_resposta)

# Definir as conexões entre os nós
grafo.add_edge(START, "entrada")
grafo.add_edge("entrada", "processamento")
grafo.add_edge("processamento", "saida")
grafo.add_edge("saida", END)

# Compilar o grafo
compiled = grafo.compile()

# Implementar ciclo de interação
while True:
    compiled.invoke({"messages": [""]})
    continuar = input("Você deseja fazer outra pergunta? (sim/não): ")
    if continuar.lower() != 'sim':
        break

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 13)

### Vídeo: Agente telecomunicações

In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, AIMessage
from typing_extensions import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langchain_community.chat_models import ChatOllama

# Inicializar o modelo
llm = ChatOllama(model="gpt-oss:20b", temperature=0)

# Definir o schema do estado
class State(TypedDict):
    messages: Annotated[list, add_messages]

# Nó de entrada
def entrada_usuario(state: State) -> State:
    pergunta = input("Pergunta: ")
    if isinstance(pergunta, str) and pergunta.strip():
        return {"messages": [HumanMessage(content=pergunta)]}
    else:
        raise ValueError("A pergunta deve ser uma string não vazia.")

# Nó de processamento usando ChatOllama
def processar_solicitacao(state: State) -> State:
    last_message = state["messages"][-1]

    if isinstance(last_message, HumanMessage) and last_message.content.strip():
        print(f"DEBUG: Processando pergunta: {last_message.content}")
        try:
            resposta = llm.invoke([last_message]).content
            print(f"DEBUG: Resposta gerada: {resposta}")
            return {"messages": [AIMessage(content=resposta)]}
        except Exception as e:
            print(f"DEBUG: Erro ao chamar o LLM: {e}")
            return {"messages": [AIMessage(content=f"Desculpe, ocorreu um erro: {e}")]}

    else:
        raise ValueError("A última mensagem no estado não é uma HumanMessage válida.")

# Nó de saída
def saida_resposta(state: State) -> State:
    last_message = state["messages"][-1]
    if hasattr(last_message, 'content'):
        print(f"Resposta: {last_message.content}")
    else:
        print(f"Resposta: {last_message}")
    return state

# Criar o grafo de estados
grafo = StateGraph(State)
grafo.add_node("entrada", entrada_usuario)
grafo.add_node("processamento", processar_solicitacao)
grafo.add_node("saida", saida_resposta)

grafo.add_edge(START, "entrada")
grafo.add_edge("entrada", "processamento")
grafo.add_edge("processamento", "saida")
grafo.add_edge("saida", END)

compiled = grafo.compile()

# Loop de interação
print("Iniciando a interação...")
while True:
    try:
        compiled.invoke({"messages": []})
        continuar = input("Deseja fazer outra pergunta? (sim/não): ")
        if continuar.lower() != "sim":
            break
    except ValueError as ve:
        print(f"Erro de entrada: {ve}")
    except KeyboardInterrupt:
        print("\nInteração encerrada pelo usuário.")
        break
    except Exception as e:
        print(f"Ocorreu um erro inesperado: {e}")
        break

print("Interação encerrada.")

/tmp/ipykernel_22381/1113382256.py:8: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  llm = ChatOllama(model="gpt-oss:20b", temperature=0)


Iniciando a interação...
Pergunta: preciso de ajuda para conectar a internet
DEBUG: Processando pergunta: preciso de ajuda para conectar a internet
DEBUG: Resposta gerada: Claro! Vamos descobrir juntos o que está acontecendo e encontrar a solução.  
Para começar, preciso de alguns detalhes:

| Pergunta | Por que é importante? |
|----------|------------------------|
| **Qual é o seu dispositivo?** (PC, laptop, smartphone, tablet, etc.) | Cada sistema tem passos ligeiramente diferentes. |
| **Qual é o sistema operacional?** (Windows 10/11, macOS Ventura, Ubuntu, Android 13, iOS 17, etc.) | As instruções variam de acordo com o SO. |
| **Você está tentando se conectar via Wi‑Fi ou cabo Ethernet?** | As configurações e os problemas comuns são diferentes. |
| **Você já tem um roteador/modem configurado?** | Se não, precisamos começar do zero. |
| **Existe algum erro ou mensagem específica que aparece?** | Isso ajuda a identificar a causa exata. |
| **Você já tentou reiniciar o roteador/modem

### Vídeo: Análise de Sentimento

In [ ]:
from langchain_community.chat_models import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage

# Inicializa o modelo ChatOllama
llm = ChatOllama(model="gpt-oss:20b", temperature=0)

def analisar_sentimento(comentario: str) -> str:
    """
    Analisa o sentimento de um comentário usando o LLM.
    Retorna 'POSITIVO', 'NEGATIVO' ou 'NEUTRO'.
    """
    prompt = (
        f"Classifique o sentimento do seguinte comentário como POSITIVO, NEGATIVO ou NEUTRO:\n"
        f"Comentário: \"{comentario}\""
    )

    # Chamando o LLM com a mensagem do usuário
    resposta = llm.invoke([HumanMessage(content=prompt)])

    # Retorna apenas o texto da resposta
    return resposta.content.strip().upper()

# Teste do sistema com um comentário de rede social
# Nota: Use Ctrl+C ou o botão de parada para encerrar o loop
try:
    while True:
        comentario = input("Digite um comentário para analisar (ou aperte stop): ")
        if not comentario: break
        sentimento = analisar_sentimento(comentario)
        print(f"O sentimento do comentário é: {sentimento}")
except KeyboardInterrupt:
    print("\nAnálise encerrada.")

Digite um comentário para analisar (ou aperte stop): mt louco!!


ConnectionError: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/chat (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7bda9eaf46b0>: Failed to establish a new connection: [Errno 111] Connection refused'))

# Pydantic AI

In [2]:
!pip install pydantic-ai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 11.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opentelemetry-instrumentation to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opentelemetry-instrumentation-httpx to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opentelemetry-sdk to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of opentelemetry-instrumentation-httpx to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver

In [ ]:
from pydantic import BaseModel, Field
from decimal import Decimal
from typing import Optional

class Produto(BaseModel):
    nome: str = Field(min_length=1, max_length=100)
    preco: Decimal = Field(gt=0, le=10000)
    descricao: Optional[str] = Field(None, max_length=500)
    categoria: str = Field(..., pattern=r'^[A-Za-z\s]+$')
    em_estoque: int = Field(ge=0)
    disponivel: bool = True

In [ ]:
try:
    produto_invalido = Produto(nome="", preco=-5, categoria="1234", em_estoque=-10)
except Exception as e:
    print("Erros de validação:", e)

Erros de validação: 4 validation errors for Produto
nome
  String should have at least 1 character [type=string_too_short, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/string_too_short
preco
  Input should be greater than 0 [type=greater_than, input_value=-5, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/greater_than
categoria
  String should match pattern '^[A-Za-z\s]+$' [type=string_pattern_mismatch, input_value='1234', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/string_pattern_mismatch
em_estoque
  Input should be greater than or equal to 0 [type=greater_than_equal, input_value=-10, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/greater_than_equal


In [ ]:
produto_valido = Produto(nome="Fone de Ouvido", preco="199.99", categoria="Eletronicos", em_estoque=15)
print(produto_valido)

nome='Fone de Ouvido' preco=Decimal('199.99') descricao=None categoria='Eletronicos' em_estoque=15 disponivel=True


In [ ]:
from pydantic import BaseModel
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIModel
from pydantic_ai.providers.ollama import OllamaProvider
import nest_asyncio

nest_asyncio.apply()

class CityLocation(BaseModel):
    city: str
    country: str

ollama_model = OpenAIModel(
    model_name='gpt-oss:20b',
    provider=OllamaProvider(base_url='http://localhost:11434/v1'),
)

agent = Agent(model=ollama_model, output_type=CityLocation)

result = agent.run_sync('Where were the Olympics held in 2012?')
print(result.output)  # -> city='London' country='United Kingdom'

/tmp/ipykernel_4968/4002991316.py:13: DeprecationWarning: `OpenAIModel` was renamed to `OpenAIChatModel` to clearly distinguish it from `OpenAIResponsesModel` which uses OpenAI's newer Responses API. Use that unless you're using an OpenAI Chat Completions-compatible API, or require a feature that the Responses API doesn't support yet like audio.
  ollama_model = OpenAIModel(


ModelAPIError: Connection error.

In [ ]:
import random
from pydantic_ai import Agent, RunContext

agent = Agent(
    model=ollama_model,
    deps_type=str,
    system_prompt="Você é um jogo de dados: role o dado e veja se acerta o palpite."
)

@agent.tool_plain
def roll_dice() -> str:
    return str(random.randint(1, 6))

@agent.tool
def get_player_name(ctx: RunContext[str]) -> str:
    return ctx.deps

dice_result = agent.run_sync('Meu palpite é 4', deps='Carlos')
print(dice_result.output)

ModuleNotFoundError: No module named 'pydantic_ai'

### Vídeo: Construção de Agentes Completos

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Protocol
from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext
from pydantic_ai.models.openai import OpenAIModel
from pydantic_ai.providers.ollama import OllamaProvider

ollama_model = OpenAIModel(
    model_name='gpt-oss:20b',
    provider=OllamaProvider(base_url='http://localhost:11434/v1'),
)


class MedicalDatabase(Protocol):
    async def get_history(self, patient_id: int) -> str: ...
    async def get_patient_name(self, patient_id: int) -> str: ...


@dataclass
class RequestContext:
    patient_id: int
    db: MedicalDatabase


class AssistanceRequest(BaseModel):
    symptoms: str = Field(max_length=500)
    urgency: int = Field(ge=1, le=5)


class AssistanceResponse(BaseModel):
    advice: str
    risk_level: int = Field(ge=0, le=10)
    refer_to_specialist: bool


assist_agent = Agent(
    model=ollama_model,
    deps_type=RequestContext,
    output_type=AssistanceResponse,
    instructions=(
        "Você é um assistente médico virtual. "
        "Avalie os sintomas de forma prudente, comunique incertezas, "
        "e forneça orientação geral e não-diagnóstica. "
        "Se houver sinais de emergência (ex.: dor torácica intensa, falta de ar importante), "
        "incentive procurar atendimento imediato (SAMU/192 ou emergência). "
        "Respeite o limite do escopo: não prescreva medicamentos."
    ),
)


@assist_agent.tool
async def fetch_patient_history(ctx: RunContext[RequestContext]) -> str:
    """Busca histórico clínico do paciente no banco."""
    return await ctx.deps.db.get_history(patient_id=ctx.deps.patient_id)


@assist_agent.system_prompt
async def add_patient_context(ctx: RunContext[RequestContext]) -> str:
    name = await ctx.deps.db.get_patient_name(ctx.deps.patient_id)
    return f"Contexto: paciente '{name}' (id={ctx.deps.patient_id})."


class FakeDB:
    async def get_history(self, patient_id: int) -> str:
        return "Hipertensão controlada; sem alergias registradas."
    async def get_patient_name(self, patient_id: int) -> str:
        return "José da Silva"

deps = RequestContext(patient_id=42, db=FakeDB())
question = AssistanceRequest(symptoms="Dor no peito e falta de ar ao esforço", urgency=5)
result = assist_agent.run(
    f"Avalie: {question.model_dump_json()}",
    deps=deps,
)
print(result.output)

/tmp/ipykernel_938/3444455258.py:10: DeprecationWarning: `OpenAIModel` was renamed to `OpenAIChatModel` to clearly distinguish it from `OpenAIResponsesModel` which uses OpenAI's newer Responses API. Use that unless you're using an OpenAI Chat Completions-compatible API, or require a feature that the Responses API doesn't support yet like audio.
  ollama_model = OpenAIModel(


AttributeError: 'coroutine' object has no attribute 'output'

### Vídeo: Multi-Agent

In [11]:
!ollama

Ollama 0.23.0

▸ Chat with a model
    Start an interactive chat with a model

  Launch Claude Code (not installed)
    Anthropic's coding tool with subagents

  Launch OpenClaw (install)
    Personal AI with 100+ skills

  Launch Hermes Agent (install)
    Self-improving AI agent built by Nous Research

  Launch OpenCode (not installed)
    Anomaly's open-source coding agent

  More...
    Show additional integrations


Error: run launcher menu: error running TUI: program was killed: program was interrupted


In [13]:
from pydantic_ai import Agent, RunContext
from pydantic_ai.usage import UsageLimits
import asyncio

joke_selection_agent = Agent(
    ollama_model,
    system_prompt='Use a `joke_factory` para gerar piadas, depois escolha a melhor e retorne apenas uma.'
)

joke_generation_agent = Agent(
    ollama_model,
    output_type=list[str]
)

@joke_selection_agent.tool
async def joke_factory(ctx: RunContext[None], count: int) -> list[str]:
    r = await joke_generation_agent.run(
        f'Por favor, gere {count} piadas.',
        usage=ctx.usage,
    )
    return r.output

async def main():
    result = await joke_selection_agent.run(
        'Conte uma piada.',
        usage_limits=UsageLimits(request_limit=5, total_tokens_limit=2000),
    )
    print(result.output)
    print(result.usage())

await main()

ModelAPIError: Connection error.

### Vídeo: Teste de Agente

In [ ]:
from pydantic_evals import Case, Dataset
from pydantic_evals.evaluators.common import IsInstance

case1 = Case(
    name='capital_franca',
    inputs='Qual é a capital da França?',
    expected_output='Paris',
    metadata={'tipo': 'fácil'},
)

dataset = Dataset(cases=[case1])
dataset.add_evaluator(IsInstance(type_name='str'))

# Mais lógica de execução e relatório...

In [12]:
from dataclasses import dataclass
from datetime import date
from pydantic_ai import Agent, RunContext
from pydantic_ai.models.test import TestModel
import asyncio

@dataclass
class MockDeps:
    weather_api: object | None = None


weather_agent = Agent(
    TestModel(),
    deps_type=MockDeps,
    system_prompt="Stubbed weather agent for testing",
)

@weather_agent.tool
def run_weather_forecast(ctx: RunContext[MockDeps], city: str = "São Paulo", when: date | None = None) -> str:
    d = (when or date.today()).isoformat()
    return f"Previsão fake para {city} em {d}: céu limpo."

async def test_weather_agent_simple():
    deps = MockDeps()

    with weather_agent.override(model=TestModel()):
        result = await weather_agent.run("Como está o tempo em SP?", deps=deps)
    assert isinstance(result.output, str)
    assert "Previsão fake" in result.output

#testando
await test_weather_agent_simple()

#### Graphs

In [16]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Optional
from pydantic_graph import BaseNode, Graph, GraphRunContext, End
import asyncio


# --------- State & Deps -------------------------------------------------

@dataclass
class WeatherState:
    query: str
    location: Optional[str] = None
    units: str = "metric"
    forecast: Optional[str] = None


@dataclass
class Deps:
    # In real code this could be an SDK client; here we just stub it.
    def fetch_weather(self, location: str, units: str) -> str:
        return f"{location}: 26° and clear ({units})"


# --------- Nodes --------------------------------------------------------

@dataclass
class DecideNext(BaseNode[WeatherState]):
    async def run(self, ctx: GraphRunContext[WeatherState]) -> AskForLocation | FetchForecast:
        if not ctx.state.location:
            return AskForLocation(prompt="Qual cidade?")
        return FetchForecast()

@dataclass
class ParseQuery(BaseNode[WeatherState]):
    async def run(self, ctx: GraphRunContext[WeatherState]) -> DecideNext:
        if "sp" in ctx.state.query.lower():
            ctx.state.location = "São Paulo"
        return DecideNext()


@dataclass
class AskForLocation(BaseNode[WeatherState, Deps]):
    prompt: str

    async def run(self, ctx: GraphRunContext[WeatherState, Deps]) -> End[str] | FetchForecast:
        ctx.state.location = ctx.state.location or "São Paulo"
        return FetchForecast()


@dataclass
class FetchForecast(BaseNode[WeatherState, Deps]):
    async def run(self, ctx: GraphRunContext[WeatherState, Deps]) -> FormatSummary:
        assert ctx.state.location  # guaranteed by DecideNext/AskForLocation
        ctx.state.forecast = ctx.deps.fetch_weather(ctx.state.location, ctx.state.units)
        return FormatSummary()


@dataclass
class FormatSummary(BaseNode[WeatherState, Deps, str]):
    async def run(self, ctx: GraphRunContext[WeatherState, Deps]) -> End[str]:
        text = f"Previsão para {ctx.state.location}: {ctx.state.forecast}"
        return End(text)


# --------- Build and run the graph -------------------------------------

weather_graph = Graph[WeatherState, Deps, str](
    nodes=[ParseQuery, DecideNext, AskForLocation, FetchForecast, FormatSummary]
)

async def run_weather_graph_example():
    result = await weather_graph.run(
        ParseQuery(),
        state=WeatherState(query="Como está o tempo em SP?"),
        deps=Deps(),
    )
    print(result.output)  # -> "Previsão para São Paulo: São Paulo: 26° and clear (metric)"

await run_weather_graph_example()

Previsão para São Paulo: São Paulo: 26° and clear (metric)


/usr/lib/python3.12/tarfile.py:56: RuntimeWarning: coroutine 'AbstractAgent.run' was never awaited
  import grp


### Vídeo: Agentes Multimodais